**[Source]** Donghwan Project (Validation 평가 지표: Balanced EM, CER, chrF 등) + Jisoo Project 11단계(ROUGE·과교정/교정누락 진단)
**[Status]** ADAPTED
**[Role]** 선택된 모델·디코딩(pko-T5, Greedy)의 전체 Validation 평가. 두 프로젝트의 지표를 함께 계산하고 문서 단위 CI를 붙임. Train에 같은 입력이 있는 행과 없는 행을 분리해 암기 효과 점검
**[Modification]** 동환의 지표 정의는 유지하되 지수 데이터(전체 Validation 54,730행)에 적용. 지표 계산 코드는 src/ko_metrics.py.
**Test는 열지 않는다.**

# 11. 최종 후보 모델의 Validation 평가
- 대상: pko-T5(paust/pko-t5-base), 전체 Train 1 epoch, Greedy — 09번에서 재사용 검증된 체크포인트의 예측.
- 기준선: **입력 그대로 복사**(Balanced EM = 0.5). 모델이 이보다 나은지 먼저 확인한다.
- 05번에서 Validation 입력의 약 17%가 Train 입력과 동일함이 확인되었다(짧은 채팅 문장). 이 행은 ‘암기’로 맞힐 수 있으므로 **Train에 없는 입력**에서의 성능을 따로 본다.
- 이 값은 **Validation 결과**이며 최종 성능이 아니다.

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 데이터·예측 로드
VAL = common.read_split(P, "validation", columns=["document_id", "utterance_id", "input", "target", "flag_seen_input_in_train", "flag_seen_pair_in_train"])
SRC, TGT, DOC = [r["input"] for r in VAL], [r["target"] for r in VAL], [r["document_id"] for r in VAL]
rows = common.read_jsonl(P.J_OUT / "decoding_comparison" / "val_predictions_greedy.jsonl")
assert [r["utterance_id"] for r in rows] == [r["utterance_id"] for r in VAL] and [r["input"] for r in rows] == SRC and [r["target"] for r in rows] == TGT
PRED = [str(r["prediction"]) for r in rows]; DEC10 = json.loads((P.RUNS / "decoding_comparison_10.json").read_text(encoding="utf-8"))
assert DEC10["selected_decoding"] == "greedy", "10번 결과와 이 노트북의 예측(Greedy)이 다름"
# flag_seen_input_in_train은 지수 데이터의 컬럼이지만, 05번 검증에서 Train 입력 집합과 직접 대조해 재확인한 값이 있는지 먼저 본다
TR_IN = set(r["input"] for r in common.read_split(P, "train", columns=["input"]))
seen = np.array([s in TR_IN for s in SRC]); flag = np.array([bool(r["flag_seen_input_in_train"]) for r in VAL])
print("Validation 입력이 Train 입력과 동일한 행(직접 계산): %d (%.2f%%) | 지수 flag와 일치: %s" % (seen.sum(), 100 * seen.mean(), bool((seen == flag).all())))
print("행 수:", len(VAL), "| 문서 수:", len(set(DOC)))

Validation 입력이 Train 입력과 동일한 행(직접 계산): 9301 (16.99%) | 지수 flag와 일치: True
행 수: 54730 | 문서 수: 1527


In [3]:
# [셀 2] 전체 Validation 지표(+ 입력 복사 기준선) — NFKC / raw
need = np.array([s != t for s, t in zip(SRC, TGT)]); docs = np.array(DOC)
ROWS = {"model": km.score_rows(PRED, TGT), "copy": km.score_rows(SRC, TGT)}
FR = json.loads((P.J_OUT / "pkot5_full" / "pkot5_full_results.json").read_text(encoding="utf-8"))
assert abs(float(ROWS["model"]["R2_어절"].mean()) - FR["rouge_mean"]["pko-T5(full)"]["R2_어절"]) < 1e-3, "지수 전체 Validation R2 재현 실패"
MET = {(k, nf): km.generation_metrics(SRC, TGT, PRED if k == "model" else SRC, nfkc=(nf == "NFKC"), with_chrf=(nf == "NFKC")) for k in ("model", "copy") for nf in ("NFKC", "raw")}
cols = ["exact_match", "need_correction_em", "unchanged_em", "balanced_em", "cer", "chrf", "rouge2_donghwan", "over_correction_rate", "miss_rate"]
print(pd.DataFrame({f"{k}|{nf}": v for (k, nf), v in MET.items()}).T[cols].round(4).to_string())
print("\n지수 ROUGE(어절/문자, 전체):")
print(pd.DataFrame({k: {m: float(v.mean()) for m, v in ROWS[k].items()} for k in ROWS}).round(4).to_string())

            exact_match  need_correction_em  unchanged_em  balanced_em     cer    chrf  rouge2_donghwan  over_correction_rate  miss_rate
model|NFKC       0.6965              0.6693        0.8707       0.7700  0.0355  0.9389           0.8080                0.1293     0.0289
model|raw        0.5269              0.5129        0.6169       0.5649  0.1025     NaN           0.7073                0.3831     0.0275
copy|NFKC        0.1351              0.0000        1.0000       0.5000  0.1578  0.7928           0.2823                0.0000     1.0000
copy|raw         0.1346              0.0000        1.0000       0.5000  0.1591     NaN           0.2818                0.0000     1.0000

지수 ROUGE(어절/문자, 전체):
          model    copy
R1_어절  0.7821  0.4216
R2_어절  0.7823  0.3969
RL_어절  0.7821  0.4216
R1_문자  0.8917  0.8762
R2_문자  0.8603  0.7458
RL_문자  0.8911  0.8745
exact    0.5269  0.1346


In [4]:
# [셀 3] 문서 단위 bootstrap 95% CI (Validation 표본 변동만 반영, 학습 seed 변동은 미반영)
ex = np.array([km.normalize_for_evaluation(p) == km.normalize_for_evaluation(t) for p, t in zip(PRED, TGT)])
ci = {}
ci["어절 R2"] = km.boot_mean(ROWS["model"]["R2_어절"], docs)
ci["문자 R2"] = km.boot_mean(ROWS["model"]["R2_문자"], docs)
ci["Balanced EM (NFKC)"] = km.boot_balanced_em(ex, need, docs)
ci["Exact Match (NFKC)"] = km.boot_mean(ex.astype(float), docs)
ci["교정 필요 행 EM (NFKC)"] = km.boot_mean(ex[need].astype(float), docs[need])
ci["원문 유지 행 EM (NFKC)"] = km.boot_mean(ex[~need].astype(float), docs[~need])
ci["기준선 대비 어절 R2 차이(모델−복사)"] = km.boot_paired_diff(ROWS["model"]["R2_어절"], ROWS["copy"]["R2_어절"], docs)
print(pd.DataFrame(ci, index=["추정", "CI95 하한", "CI95 상한"]).T.round(4).to_string())

                                       추정  CI95 하한  CI95 상한
어절 R2                              0.7823     0.7776     0.7868
문자 R2                              0.8603     0.8567     0.8643
Balanced EM (NFKC)                   0.7701     0.7618     0.7778
Exact Match (NFKC)                   0.6965     0.6908     0.7023
교정 필요 행 EM (NFKC)               0.6694     0.6636     0.6754
원문 유지 행 EM (NFKC)               0.8709     0.8549     0.8848
기준선 대비 어절 R2 차이(모델−복사)  0.3854     0.3799     0.3915


In [5]:
# [셀 4] Train 입력과 겹치는 행 vs 겹치지 않는 행 (암기 효과 점검)
out = {}
for name, mask in (("Train에 같은 입력 있음", seen), ("Train에 같은 입력 없음", ~seen)):
    idx = np.where(mask)[0]; m = km.generation_metrics([SRC[i] for i in idx], [TGT[i] for i in idx], [PRED[i] for i in idx], with_chrf=False)
    m["어절 R2"] = float(ROWS["model"]["R2_어절"][idx].mean()); m["기준선(복사) 어절 R2"] = float(ROWS["copy"]["R2_어절"][idx].mean()); m["평균 입력 길이(문자)"] = float(np.mean([len(SRC[i]) for i in idx])); out[name] = m
print(pd.DataFrame(out).T[["n", "n_need_correction", "n_unchanged", "평균 입력 길이(문자)", "exact_match", "need_correction_em", "unchanged_em", "balanced_em", "cer", "어절 R2", "기준선(복사) 어절 R2", "over_correction_rate", "miss_rate"]].round(4).to_string())
ns = np.where(~seen)[0]
b = km.boot_balanced_em(ex[ns], need[ns], docs[ns]); print("\nTrain에 없는 입력만의 Balanced EM(NFKC): %.4f (CI %.4f~%.4f)" % b)

                              n  n_need_correction  n_unchanged  평균 입력 길이(문자)  exact_match  need_correction_em  unchanged_em  balanced_em     cer  어절 R2  기준선(복사) 어절 R2  over_correction_rate  miss_rate
Train에 같은 입력 있음   9301.0             5964.0       3337.0                4.3767       0.8021              0.7307        0.9296       0.8301  0.0476   0.9124                0.8186                0.0704     0.0491
Train에 같은 입력 없음  45429.0            41370.0       4059.0               15.5478       0.6749              0.6604        0.8224       0.7414  0.0348   0.7557                0.3105                0.1776     0.0260

Train에 없는 입력만의 Balanced EM(NFKC): 0.7414 (CI 0.7292~0.7524)


In [6]:
# [셀 5] 입력 길이별 성능 · 그림
lens = np.array([len(s) for s in SRC]); bins = [(0, 5), (6, 10), (11, 20), (21, 40), (41, 10**6)]; lab = ["≤5", "6-10", "11-20", "21-40", ">40"]
tab = []
for (lo, hi), l in zip(bins, lab):
    idx = np.where((lens >= lo) & (lens <= hi))[0]
    tab.append({"입력 길이(문자)": l, "행 수": len(idx), "교정 필요 비율": float(need[idx].mean()), "EM(NFKC)": float(ex[idx].mean()), "어절 R2": float(ROWS["model"]["R2_어절"][idx].mean()), "복사 어절 R2": float(ROWS["copy"]["R2_어절"][idx].mean())})
LT = pd.DataFrame(tab); print(LT.round(4).to_string(index=False))
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.5, 3.8)); x = np.arange(len(lab)); ax.bar(x - 0.2, LT["EM(NFKC)"], 0.4, label="Model EM (NFKC)"); ax.bar(x + 0.2, LT["교정 필요 비율"], 0.4, label="Share needing correction")
ax.set_xticks(x); ax.set_xticklabels(lab); ax.set_xlabel("Input length (characters)"); ax.legend(); ax.set_title("Validation by input length")
plt.tight_layout(); plt.savefig(P.REPORTS / "fig11_by_length.png", dpi=150); plt.close()
(P.RUNS / "validation_eval_11.json").write_text(json.dumps({"metrics": {f"{k}|{nf}": v for (k, nf), v in MET.items()}, "ci95": {k: list(map(float, v)) for k, v in ci.items()}, "seen_vs_unseen": out, "by_length": LT.to_dict("records"), "scope": "pko-T5 full, Greedy, 전체 Validation, Test 미사용"}, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("저장: runs/validation_eval_11.json, reports/fig11_by_length.png")

입력 길이(문자)  행 수  교정 필요 비율  EM(NFKC)  어절 R2  복사 어절 R2
             ≤5  11812          0.7541    0.7322   0.8716        0.7224
           6-10  14077          0.8490    0.7396   0.7227        0.2327
          11-20  18407          0.8986    0.7011   0.7662        0.3124
          21-40   8979          0.9524    0.6137   0.7897        0.3913
            >40   1455          0.9704    0.4419   0.7927        0.4453
저장: runs/validation_eval_11.json, reports/fig11_by_length.png


## 해석
- **기준선 대비**: 입력 복사(Balanced EM 0.500, 어절 R2 0.397) 대비 모델은 Balanced EM(NFKC) 0.770(CI95 0.762~0.778), 어절 R2 0.782(0.778~0.787), 기준선과의 R2 차이 +0.385(0.380~0.392)로 명확히 높다. 지수 11·12단계의 ROUGE와 소수 4자리까지 재현되었다.
- **NFKC와 raw의 큰 차이**: raw 기준 Exact Match는 52.7%, Balanced EM 0.565이고 NFKC 기준은 69.7%, 0.770이다. 차이(약 17%p)는 모델의 실제 교정 실력이 아니라 tokenizer 정규화로 문자가 바뀐 것에서 온다(12번에서 원인 분석·후처리 검증). **raw 값과 NFKC 값 중 어느 하나만 보고하면 안 된다.**
- **암기 효과 점검**: Validation의 17.0%(9,301행)는 Train에 같은 입력이 있고 평균 4.4자의 짧은 채팅 문장이다. 이 행의 Balanced EM은 0.830, Train에 없는 입력은 0.741(CI 0.729~0.752)이다. 두 부분집합의 문장 길이와 유형이 크게 달라 차이를 전부 ‘암기’ 때문이라고 단정할 수는 없지만, **Train에 없는 입력만으로도 기준선(0.5)보다 명확히 높다**는 것은 확인된다. 전체 점수를 일반화 성능으로 읽을 때는 Train에 없는 입력 기준 값을 함께 보고해야 한다.
- **입력이 길수록 성능이 떨어진다**: EM(NFKC)은 ≤5자 0.732, 6-10자 0.740, 11-20자 0.701, 21-40자 0.614, 40자 초과 0.442. 어절 R2는 길어져도 0.79 수준으로 유지되어 EM이 문장 전체 일치라는 엄격한 지표라는 점과 일치한다(원인이 길이 자체인지 긴 문장에 오류가 더 많이 섞이기 때문인지는 분리하지 않았다).
- **교정 필요 행이 91.6%**(원문 유지 7,396행, 13.5%)이므로 일반 EM은 ‘교정 필요 행 EM’이 거의 지배한다. Balanced EM은 원문 유지 행을 절반 가중치로 넣는 지표라서 원문 유지 행의 표본이 작은 만큼 CI가 넓다(원문 유지 EM CI 0.855~0.885).
- **한계**: CI는 Validation 표본 변동만 반영하고 학습 seed 변동은 반영하지 않는다(seed 1회). 이 값은 Validation 결과이며 최종(Test) 성능이 아니다.